In [ ]:
import os


In [14]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
make_guest_centers_plotly.py
 ├─ re-imaged XYZ → guest 중심 좌표 추출
 ├─ K-means 군집화
 └─ Plotly 3-D 시각화 (축 눈금 1 Å 간격, 동등 스케일)

pip install numpy scikit-learn plotly
"""
import os, glob
import numpy as np
from sklearn.cluster import KMeans
import plotly.graph_objects as go
import plotly.express   as px   # 색상 팔레트용

# ---------- 사용자 파라미터 ----------
DIR_PATH      = './c'     # reimg XYZ 폴더
N_GUEST_ATOMS = 15        # guest 분자 원자 수
N_CLUSTERS    = 8         # k-means k 값
OUT_XYZ       = 'guest_centers.xyz'
HTML_OUT      = 'guest_centers_clusters.html'
# -------------------------------------

# 1) 중심 좌표 계산 -------------------------------------------------
centers = []
xyz_files = sorted(glob.glob(os.path.join(DIR_PATH, '*reimg*.xyz')))
if not xyz_files:
    raise FileNotFoundError(f'폴더 {DIR_PATH} 에 reimg XYZ 파일이 없습니다.')

for fpath in xyz_files:
    with open(fpath) as f:
        lines = f.readlines()
    atom_lines  = lines[2:]
    guest_lines = atom_lines[-N_GUEST_ATOMS:]
    coords = np.array([[float(x), float(y), float(z)]
                       for _, x, y, z, *_ in (ln.split() for ln in guest_lines)],
                      float)
    centers.append(coords.mean(axis=0))
centers = np.vstack(centers)                 # (n_mols, 3)

# 2) XYZ 파일 저장 -------------------------------------------------
with open(OUT_XYZ, 'w') as f:
    f.write(f"{len(centers)}\nguest molecule centers\n")
    for x, y, z in centers:
        f.write(f"C {x:.6f} {y:.6f} {z:.6f}\n")
print(f"[✓] 중심 좌표 XYZ 저장 → {OUT_XYZ}")

# 3) K-means -------------------------------------------------------
kmeans = KMeans(n_clusters=N_CLUSTERS, n_init='auto', random_state=1)
labels = kmeans.fit_predict(centers)
print(f"[✓] K-means 완료 (k={N_CLUSTERS})")

# 4) Plotly 3-D 산점도 --------------------------------------------
palette = px.colors.qualitative.Plotly  # 기본 10색 팔레트
colors  = [palette[i % len(palette)] for i in labels]

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=centers[:, 0], y=centers[:, 1], z=centers[:, 2],
            mode='markers',
            marker=dict(size=4, color=colors),
            text=[f"Mol {i} → Cluster {lbl+1}" for i, lbl in enumerate(labels)],
            hovertemplate='%{text}<br>x=%{x:.2f} Å<br>y=%{y:.2f} Å<br>z=%{z:.2f} Å<extra></extra>',
        )
    ]
)

# 동일 눈금 & 1 Å 간격 -----------------
axis_cfg = dict(
    showbackground=True,
    backgroundcolor='rgba(240,240,240,0.3)',
    showgrid=True,
    zeroline=True,
    dtick=1,             # 눈금 간격 1 Å
)
fig.update_layout(
    title=f'Guest centers (k={N_CLUSTERS})',
    scene=dict(
        xaxis=axis_cfg,
        yaxis=axis_cfg,
        zaxis=axis_cfg,
        aspectmode='data'   # 각 축 동일 스케일
    ),
    margin=dict(l=0, r=0, t=30, b=0),
    legend_title='Clusters'
)

fig.show()                           # 노트북/터미널에서 바로 보기
fig.write_html(HTML_OUT)             # 독립형 HTML 저장
print(f"[✓] Plotly HTML 저장 → {HTML_OUT}")


[✓] 중심 좌표 XYZ 저장 → guest_centers.xyz
[✓] K-means 완료 (k=8)


[✓] Plotly HTML 저장 → guest_centers_clusters.html
